# Extract Annotations

In [ ]:
import os
import random
# import json
from tqdm.auto import tqdm
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
sns.set_style("dark")

from shapely.geometry import box
import numpy as np

import tomllib
import avoddiag as ag

## Load Configuration

In [ ]:
with open("config/config.toml", "rb") as f:
    config = tomllib.load(f)

## Load Image Dataset

In [ ]:
image_dataset = ag.image.data.Dataset(
    root_folder_path="output/TestDataset_03_Generated+Edited_Images",
)

In [ ]:
image_dataset.metadata.keys()

In [ ]:
image_dataset.metadata['attributes_filtered:gemini-2.5-flash']

## Extract Image Annotations

### Moondream2

In [ ]:
!nvidia-smi

In [ ]:
# Initialize the Moondream provider
provider_moondream = ag.providers.transformers.Moondream2(
    gpu_ids=[0,1,2,3,4,5]
)

In [ ]:
model_name = "vikhyatk/moondream2"
model_revision = "2025-06-21"

annotation_extractor_moondream = ag.image.analysis.AnnotationExtractor(
    provider=provider_moondream,
    model_name=model_name,
    model_revision=model_revision
)

In [ ]:
object_name = "cars and pickup trucks"

annotation_extractor_moondream.extract_dataset_annotations(
    image_dataset=image_dataset,
    object_name=object_name
)

In [ ]:
image_dataset.metadata['annotations_generated:vikhyatk+moondream2:cars and pickup trucks']

### Gemini-2.5-Flash

In [ ]:
# Initialize the Google GenAI provider
provider_google = ag.providers.google.GoogleGenAI(api_key=config['providers']['google']['api_key'])

In [ ]:
models_data = provider_google.list_models()
models_data[models_data['name'] == "gemini-2.5-flash"]

In [ ]:
models_data[models_data['name'].apply(lambda x: x.find("latest")>=0)]

In [ ]:
model_name = "gemini-2.5-flash-lite"

annotation_extractor_google = ag.image.analysis.AnnotationExtractor(
    provider=provider_google,
    model_name=model_name,
    with_caching=True,
)

In [ ]:
object_name = "cars and pickup trucks"

image_file_paths_to_process = sorted(
    map(
        lambda fn: os.path.join(image_dataset.images_folder_path, fn), 
        image_dataset.metadata['attributes_filtered:gemini-2.5-flash'][image_dataset.metadata['attributes_filtered:gemini-2.5-flash']['gen:vehicle_presence']]['image_file_name'].to_list()
    )
)

annotation_extractor_google.extract_dataset_annotations(
    image_dataset=image_dataset,
    object_name=object_name,
    image_file_paths_to_process=image_file_paths_to_process
)

In [ ]:
# image_dataset.metadata['annotations_generated:gemini-2.5-flash-lite:car']
metadata_key = f'annotations_generated:{model_name}:{object_name}'
image_dataset.metadata[metadata_key]

In [ ]:
# Add images with no vehicle presence

images_metadata_empty = image_dataset.metadata['attributes_filtered:gemini-2.5-flash'][~image_dataset.metadata['attributes_filtered:gemini-2.5-flash']['gen:vehicle_presence']].copy()[
    [
        'image_file_name',
    ]
]
images_metadata_empty['response_raw'] = ''
images_metadata_empty['query_status'] = 'N/A'
images_metadata_empty['exception'] = np.nan
images_metadata_empty['elapsed_time_sec'] = np.nan
images_metadata_empty['response_parsed'] = [[] for _ in range(len(images_metadata_empty))]
images_metadata_empty['is_parsing_successful'] = True
images_metadata_empty['annotations:bbox2d_xywh_px'] = [[] for _ in range(len(images_metadata_empty))]
images_metadata_empty['has_bboxes'] = False
images_metadata_empty['num_objects'] = 0
images_metadata_empty['annotations:bbox2d_xywh_px_reduced'] = [[] for _ in range(len(images_metadata_empty))]
images_metadata_empty['num_objects_reduced'] = 0

images_metadata_empty

In [ ]:
image_dataset.metadata[metadata_key] = pd.concat([image_dataset.metadata[metadata_key], images_metadata_empty], ignore_index=True).sort_values('image_file_name').reset_index(drop=True)
image_dataset.metadata[metadata_key]

In [ ]:
image_dataset.save_metadata(keys_to_overwrite=[metadata_key])

#### Recover truncated responses

In [ ]:
# image_dataset.metadata['annotations_generated:gemini-2.5-flash-lite:car']['is_parsing_successful'].value_counts()
image_dataset.metadata[metadata_key]['is_parsing_successful'].value_counts()

In [ ]:
image_dataset.metadata[metadata_key][image_dataset.metadata[metadata_key]['is_parsing_successful'] == False]

In [ ]:
import json
from typing import Any, Callable, Dict, List, Optional, Tuple

def _default_validator(obj: Any) -> bool:
    """
    Example schema check (optional): keep only objects that look like your boxes.
    Modify/remove as needed for other schemas.
    """
    if not isinstance(obj, dict):
        return False
    if "box_2d" not in obj or "label" not in obj:
        return False
    box = obj["box_2d"]
    return (
        isinstance(box, list) and
        len(box) == 4 and
        all(isinstance(v, (int, float)) for v in box) and
        isinstance(obj["label"], str)
    )

def recover_json_objects(
    s: str,
    validate: Optional[Callable[[Any], bool]] = _default_validator
) -> Tuple[List[Dict[str, Any]], int, int]:
    """
    Recover as many complete JSON objects as possible from an incomplete string.

    Parameters
    ----------
    s : str
        The (possibly truncated) JSON-like text.
    validate : callable or None
        Optional predicate to keep/skip decoded objects. If None, keeps all
        successfully decoded objects.

    Returns
    -------
    objects : list of dict
        All successfully decoded and (optionally) validated JSON objects.
    last_complete_end : int
        Index in `s` immediately after the last fully decoded object.
        (Everything from this index onward is tail/garbage/truncated.)
    total_attempted : int
        Number of balanced object substrings we attempted to decode.
    """
    objs: List[Dict[str, Any]] = []
    i = 0
    n = len(s)

    in_string = False
    escape = False
    brace_depth = 0
    start_idx: Optional[int] = None
    last_complete_end = 0
    attempted = 0

    while i < n:
        ch = s[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch == "{":
                if brace_depth == 0:
                    start_idx = i
                brace_depth += 1
            elif ch == "}":
                if brace_depth > 0:
                    brace_depth -= 1
                    if brace_depth == 0 and start_idx is not None:
                        # We have a balanced {...} chunk
                        chunk = s[start_idx:i+1]
                        attempted += 1
                        try:
                            obj = json.loads(chunk)
                            if (validate is None) or validate(obj):
                                objs.append(obj)
                                last_complete_end = i + 1
                        except Exception:
                            pass
                        start_idx = None

        i += 1

    return objs, last_complete_end, attempted



In [ ]:

# mask = image_dataset.metadata['annotations_generated:gemini-2.5-flash-lite:car']['is_parsing_successful'] == False
metadata_key = f'annotations_generated:{model_name}:{object_name}'
mask = image_dataset.metadata[metadata_key]['is_parsing_successful'] == False

recovered_all = []
for i_row, row in tqdm(image_dataset.metadata[metadata_key][mask].iterrows(), total=mask.sum()):
    # print(f"Row {i_row}: image_file_name={row['image_file_name']}")
    truncated = row['response_raw']

    if isinstance(truncated, str):
        recovered, last_end, attempted = recover_json_objects(truncated)
        
    else:
        recovered, last_end, attempted = [], 0, 0

    # print(f"  Recovered objects: {len(recovered)} (attempted {attempted})")
    # print(f"  Tail starts at index: {last_end} (len={len(truncated)})")
    # if recovered:
        # print("  First object:", recovered[0])
        # print("  Last object:", recovered[-1])

    recovered_all.append(recovered)


In [ ]:
image_dataset.metadata[metadata_key][mask]

In [ ]:
image_dataset.metadata[metadata_key].loc[mask, 'response_parsed'] = pd.Series(recovered_all, index=image_dataset.metadata[metadata_key].loc[mask].index)

In [ ]:
image_dataset.metadata[metadata_key][mask]

In [ ]:
bbox2d_denormalized_all = []
for i_row, row in tqdm(image_dataset.metadata[metadata_key][mask].iterrows(), total=mask.sum()):
    response_parsed = row['response_parsed']
    # assert len(response_parsed) == 0

    image_data = image_dataset[row['image_file_name']]
    image = image_data['image']
    # bboxes_xywh = provider_google.bbox2d_denormalize(list(filter(lambda x: x['label'] == 'car', response_parsed)), image)
    bboxes_xywh = provider_google.bbox2d_denormalize(response_parsed, image.size)
    # image_dataset.metadata.at[i_row, 'response_parsed'] = response_parsed
    bbox2d_denormalized_all.append(bboxes_xywh)

In [ ]:
image_dataset.metadata[metadata_key]['response_parsed'].explode().dropna().apply(lambda x: x['label']).value_counts()

In [ ]:
image_dataset.metadata[metadata_key].loc[mask, 'annotations:bbox2d_xywh_px'] = pd.Series(bbox2d_denormalized_all, index=image_dataset.metadata[metadata_key].loc[mask].index)

In [ ]:
image_dataset.metadata[metadata_key][mask]

In [ ]:
image_dataset.save_metadata(keys_to_overwrite=[metadata_key])

### Visualize Samples

In [ ]:
# idx = 0
idx = random.randint(0, len(image_dataset) - 1)
image_data = image_dataset[idx]

image = image_data['image']


model_name = "gemini-2.5-flash-lite"
object_name = "cars and pickup trucks"
metadata_key = ':'.join(
    [
        'annotations_generated',
        f'{model_name.replace("/", "+")}',
        f'{object_name.replace("/", "+")}'
    ]
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image)

# for l, t, w, h in image_data['metadata'][metadata_key]['annotations:bbox2d_xywh_px_reduced']:
for l, t, w, h in image_data['metadata'][metadata_key]['annotations:bbox2d_xywh_px']:
    rect = patches.Rectangle((l, t), w, h, linewidth=1, edgecolor='red', facecolor='none')
    ax.add_patch(rect)

ax.axis('off')
ax.set_title(f"Image ID: {image_data['image_file_name']} | Model: {model_name} | Object: {object_name}")

In [ ]:
idx = random.randint(0, len(image_dataset) - 1)
image_data = image_dataset[idx]

image = image_data['image']

# colors = sns.color_palette("hls", 2)
colors = sns.color_palette("muted", 2)
# colors = ['red', 'lime']
model_names = ["gemini-2.5-flash-lite", "vikhyatk/moondream2"]

object_name = "cars and pickup trucks"

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image)

num_bboxes_all = 0
for i_model, model_name in enumerate(model_names):
    metadata_key = ':'.join(
        [
            'annotations_generated',
            f'{model_name.replace("/", "+")}',
            f'{object_name.replace("/", "+")}'
        ]
    )

    bboxes = image_data['metadata'][metadata_key]['annotations:bbox2d_xywh_px']
    print(f'Model: {model_name}, Number of boxes: {len(bboxes):,d}')
    num_bboxes_all += len(bboxes)
    for i, (l, t, w, h) in enumerate(bboxes):
        rect = patches.Rectangle(
            (l, t), w, h, 
            linewidth=len(model_names) - i_model,
            edgecolor=colors[i_model],
            facecolor='none',
            # alpha=0.7,
            label=model_name if i == 0 else None
        )

        ax.add_patch(rect)

ax.axis('off')

if num_bboxes_all > 0:
    ax.legend(title='Model')

ax.set_title(f"Image ID: {image_data['image_file_name']} | Object: {object_name}\nVehicle Presence: {image_data['metadata']['attributes_filtered:gemini-2.5-flash']['gen:vehicle_presence']}")

## Filter Bounding Boxes

Methodology:
- Check whether there are images with no vehicle presence but generated bboxes and vise versa
- Perform in-model IoU-based bbox filtering
- Perform cross-model IoU-based matching and annotaitons consensus estimation
- Analyze images with bad consensus
    - Object size may be a reason for bad annotations
    - If not many images have problem, just reject them 


### Image-wise in-model IoU-based bboxes filtering

In [ ]:
def get_iou(bbox1, bbox2):
    intersection = bbox1.intersection(bbox2)
    union = bbox1.union(bbox2)

    if union.area == 0:
        return 0.0

    iou = intersection.area / union.area
    return iou


# Calculate IoUs between all boxes
def get_bboxes_ious(bboxes_shp):
    indices = []
    for i in range(len(bboxes_shp)):
        for j in range(i + 1, len(bboxes_shp)):
            indices.append((i, j))

    ious = np.zeros((len(bboxes_shp), len(bboxes_shp)))
    for i, j in indices:
        iou_val = get_iou(bboxes_shp[i], bboxes_shp[j])
        ious[i,j] = iou_val
        ious[j,i] = iou_val

    return ious




model_names = ["vikhyatk/moondream2", "gemini-2.5-flash-lite"]
object_name = "cars and pickup trucks"

for model_name in model_names:
    metadata_key = f'annotations_generated:{model_name.replace("/", "+")}:{object_name}'

    annotations_metadata = image_dataset.metadata[metadata_key]

    annotations_metadata['num_objects'] = annotations_metadata['annotations:bbox2d_xywh_px'].apply(len)
    annotations_metadata['has_bboxes'] = annotations_metadata['annotations:bbox2d_xywh_px'].apply(len) > 0
    annotations_metadata[['image_file_name', 'has_bboxes']]


    bboxes_xywh_reduced_all = []
    for i_row, row in tqdm(annotations_metadata[annotations_metadata['has_bboxes']].iterrows(), total=annotations_metadata['has_bboxes'].sum(), desc=f"Processing model: {model_name}"):
        # print(row['image_file_name'], len(row['annotations:bbox2d_xywh_px']))

        # Convert bboxes to shaply box
        bboxes_shp = []
        for bbox_xywh in row['annotations:bbox2d_xywh_px']:
            bbox = box(
                bbox_xywh[0],
                bbox_xywh[1],
                bbox_xywh[0] + bbox_xywh[2],
                bbox_xywh[1] + bbox_xywh[3],
            )
            # print(bbox)
            bboxes_shp.append(bbox)
            # break

        bboxes_shp = np.array(bboxes_shp)

        ious = get_bboxes_ious(bboxes_shp)
        # print(ious)

        mask_iou = ious >= 0.5
        np.fill_diagonal(mask_iou, 1)
        

        bbox_sets = []
        for i in range(len(bboxes_shp)):
            bbox_sets.append(tuple(sorted(mask_iou[i].nonzero()[0])))

        bbox_sets_reduced = list(set(bbox_sets))
        bboxes_xywh_reduced = []
        for s in bbox_sets_reduced:
            # Compute the mean bounding box from the elements in bboxes_shp[list(s)]
            selected_boxes = [bboxes_shp[i] for i in list(s)]
            if selected_boxes:
                # Get bounds for each box: (minx, miny, maxx, maxy)
                bounds = np.array([box.bounds for box in selected_boxes])
                mean_minx = bounds[:, 0].mean()
                mean_miny = bounds[:, 1].mean()
                mean_maxx = bounds[:, 2].mean()
                mean_maxy = bounds[:, 3].mean()
                mean_bbox = box(mean_minx, mean_miny, mean_maxx, mean_maxy)

            else:
                mean_bbox = None

            assert mean_bbox is not None, "This should not happen"
            
            x_min, y_min, x_max, y_max = mean_bbox.bounds
            bboxes_xywh_reduced.append([x_min, y_min, x_max - x_min, y_max - y_min])

        bboxes_xywh_reduced_all.append(bboxes_xywh_reduced)


    annotations_metadata['annotations:bbox2d_xywh_px_reduced'] = [[] for _ in range(len(annotations_metadata))]
    annotations_metadata.loc[annotations_metadata['has_bboxes'], 'annotations:bbox2d_xywh_px_reduced'] = pd.Series(bboxes_xywh_reduced_all, index=annotations_metadata[annotations_metadata['has_bboxes']].index)
    annotations_metadata['num_objects_reduced'] = annotations_metadata['annotations:bbox2d_xywh_px_reduced'].apply(lambda x: len(x) if isinstance(x, list) else 0)

    print(annotations_metadata[['num_objects', 'num_objects_reduced']].sum())

    image_dataset.save_metadata(keys_to_overwrite=[metadata_key])

### Cross-model annotations matching

In [ ]:
model_names = ["gemini-2.5-flash-lite", "vikhyatk/moondream2"]
object_name = "cars and pickup trucks"

metadata_keys = []
for model_name in model_names:
    metadata_key = f'annotations_generated:{model_name.replace("/", "+")}:{object_name}'
    metadata_keys.append(metadata_key)


In [ ]:
annotations_metadata = image_dataset.metadata[metadata_keys[0]][['image_file_name', 'annotations:bbox2d_xywh_px_reduced']].merge(
    image_dataset.metadata[metadata_keys[1]][['image_file_name', 'annotations:bbox2d_xywh_px_reduced']],
    on='image_file_name',
    suffixes=(f':{model_names[0]}', f':{model_names[1]}')
).merge(
    image_dataset.metadata['attributes_filtered:gemini-2.5-flash'][['image_file_name', 'gen:vehicle_presence']],
    on='image_file_name'
)
# annotations_metadata

In [ ]:
def convert_bbox2d_to_shapely_format(bboxes_xywh):
    bboxes_shp = []
    for bbox_xywh in bboxes_xywh:
        bbox = box(
            bbox_xywh[0],
            bbox_xywh[1],
            bbox_xywh[0] + bbox_xywh[2],
            bbox_xywh[1] + bbox_xywh[3],
        )
        # print(bbox)
        bboxes_shp.append(bbox)
        # break
    return np.array(bboxes_shp)

def get_iou(bbox1, bbox2):
    intersection = bbox1.intersection(bbox2)
    union = bbox1.union(bbox2)

    if union.area == 0:
        return 0.0

    iou = intersection.area / union.area
    return iou


# Calculate IoUs between all boxes
def get_bboxes_ious(bboxes_shp_A, bboxes_shp_B):
    indices = []
    for i in range(len(bboxes_shp_A)):
        for j in range(len(bboxes_shp_B)):
            indices.append((i, j))

    ious = np.zeros((len(bboxes_shp_A), len(bboxes_shp_B)))
    for i, j in indices:
        iou_val = get_iou(bboxes_shp_A[i], bboxes_shp_B[j])
        ious[i,j] = iou_val

    return ious


model_names_str = ", ".join([f'"{m}"' for m in model_names])
print(f'Model names: {model_names_str}')

annotations_matched = []
annotations_union   = [] 
for i_row, row in tqdm(annotations_metadata.iterrows(), total=len(annotations_metadata)):
    # print(row['image_file_name'], len(row['annotations:bbox2d_xywh_px:gemini-2.5-flash-lite']), len(row['annotations:bbox2d_xywh_px:vikhyatk/moondream2']), row['gen:vehicle_presence'])
    
    # Convert bboxes to shaply box
    bboxes_shp_A = convert_bbox2d_to_shapely_format(row[f'annotations:bbox2d_xywh_px_reduced:{model_names[0]}'])
    bboxes_shp_B = convert_bbox2d_to_shapely_format(row[f'annotations:bbox2d_xywh_px_reduced:{model_names[1]}'])

    ious = get_bboxes_ious(bboxes_shp_A, bboxes_shp_B)
    mask_ious = (ious >= 0.5)

    bboxes_A_indices, bboxes_B_indices = mask_ious.nonzero()
    assert bboxes_A_indices.shape[0] == bboxes_B_indices.shape[0]

    # if len(bboxes_shp_A) > 0 and len(bboxes_shp_B) > 0:
    #     assert False

    bboxes_xywh_mean_all = []
    for idx_A, idx_B in zip(bboxes_A_indices, bboxes_B_indices):
        bbox_shp_A = bboxes_shp_A[idx_A]
        bbox_shp_B = bboxes_shp_B[idx_B]

        bounds = np.array([box.bounds for box in [bbox_shp_A, bbox_shp_B]])
        mean_minx = bounds[:, 0].mean()
        mean_miny = bounds[:, 1].mean()
        mean_maxx = bounds[:, 2].mean()
        mean_maxy = bounds[:, 3].mean()

        bboxes_xywh_mean_all.append([mean_minx, mean_miny, mean_maxx - mean_minx, mean_maxy - mean_miny])

    annotations_matched.append(bboxes_xywh_mean_all)

    # 
    image_annotations_union = []
    image_annotations_union.extend(bboxes_xywh_mean_all)

    for i, bbox in enumerate(bboxes_shp_A[~mask_ious.any(axis=1)]):
        image_annotations_union.append([bbox.bounds[0], bbox.bounds[1], bbox.bounds[2] - bbox.bounds[0], bbox.bounds[3] - bbox.bounds[1]])

    for i, bbox in enumerate(bboxes_shp_B[~mask_ious.any(axis=0)]):
        image_annotations_union.append([bbox.bounds[0], bbox.bounds[1], bbox.bounds[2] - bbox.bounds[0], bbox.bounds[3] - bbox.bounds[1]])

    annotations_union.append(image_annotations_union)

In [ ]:
annotations_metadata['annotations:bbox2d_xywh_px_reduced_matched'] = pd.Series(annotations_matched, index=annotations_metadata.index)
annotations_metadata['annotations:bbox2d_xywh_px_reduced_union'] = pd.Series(annotations_union, index=annotations_metadata.index)
annotations_metadata.drop(columns=['annotations_overlapped', 'annotations_matched'], inplace=True, errors='ignore')
# annotations_metadata

In [ ]:
metadata_key = f'annotations_matched+union:{"_".join(model_names).replace("/", "+")}:{object_name.replace("/", "+")}'
print(metadata_key)

image_dataset.metadata[metadata_key] = annotations_metadata

### Store Metadata

In [ ]:
# Store
image_dataset.save_metadata(keys_to_overwrite=[metadata_key])

### Visualize Results

In [ ]:
# Visualize results
metadata_key = 'annotations_matched+union:gemini-2.5-flash-lite_vikhyatk+moondream2:cars and pickup trucks'

print(metadata_key)
image_metadata = image_dataset.metadata[metadata_key]
# image_metadata

In [ ]:
# d_image = image_metadata[image_metadata['gen:vehicle_presence']].sample().iloc[0]

image_file_name = d_image['image_file_name']
image_data = image_dataset[image_file_name]

image = image_data['image']


# colors = sns.color_palette("hls", 2)
colors = sns.color_palette("muted", 2)
# colors = ['red', 'lime']


fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image)

metadata_keys = [
    'annotations:bbox2d_xywh_px_reduced:gemini-2.5-flash-lite',
    'annotations:bbox2d_xywh_px_reduced:vikhyatk/moondream2',
    'annotations:bbox2d_xywh_px_reduced_union'
]

for metadata_key, color, label in zip(metadata_keys, colors + ['lime'], ['gemini-2.5-flash-lite', 'vikhyatk/moondream2', 'union']):
    bboxes = d_image[metadata_key]
    num_bboxes_all = len(bboxes)
    print(f'Metadata key: {metadata_key}, Number of boxes: {num_bboxes_all:,d}')
    for i, (l, t, w, h) in enumerate(bboxes):
        rect = patches.Rectangle(
            (l, t), w, h, 
            linewidth=1,
            linestyle='--' if label == 'union' else 'solid',
            edgecolor=color,
            facecolor='none',
            # alpha=0.7,
            label=label if i == 0 else None
        )

        ax.add_patch(rect)

ax.axis('off')
ax.legend(title='Model')

# ax.set_title(f"Image ID: {image_data['image_file_name']} | Object: {object_name}\nVehicle Presence: {image_data['metadata']['attributes_filtered:gemini-2.5-flash']['gen:vehicle_presence']}")

## Convert to COCO format

In [ ]:
metadata_key = 'annotations_matched+union:gemini-2.5-flash-lite_vikhyatk+moondream2:cars and pickup trucks'

images_metadata = image_dataset.metadata[metadata_key]
images_metadata

### Original Bounding Boxes

In [ ]:
coco_object_categories = pd.DataFrame(
    [
        {'id': 1, 'name': 'small'}
    ]
).set_index('id').sort_index()

coco_annotations = {
    'categories': coco_object_categories.reset_index().to_dict(orient='records'),
    'images': [],
    'annotations': []
}

for i_row, row in tqdm(images_metadata.iterrows(), total=len(images_metadata)):
    image_id = len(coco_annotations['images'])

    image_data = image_dataset[row['image_file_name']]
    image = image_data['image']

    coco_image = {
        'id': image_id,
        'file_name': row['image_file_name'],
        'width': image.width,
        'height': image.height
    }
    coco_annotations['images'].append(coco_image)

    # for bbox_xywh in row['annotations:bbox2d_xywh_px_reduced_matched']:
    for bbox_xywh in row['annotations:bbox2d_xywh_px_reduced_union']:
        annotation_id = len(coco_annotations['annotations'])
        coco_annotation = {
            'id': annotation_id,
            'image_id': image_id,
            'category_id': 1,
            'bbox': [round(v, 2) for v in bbox_xywh],
            'area': round(bbox_xywh[2] * bbox_xywh[3], 2),
            'iscrowd': 0
        }
        coco_annotations['annotations'].append(coco_annotation)



# Save annotations
annotations_folder_path = os.path.join(image_dataset.root_folder_path, 'annotations_coco')
os.makedirs(annotations_folder_path, exist_ok=True)

annotations_file_path = os.path.join(
    annotations_folder_path,
    # 'matched_gemini-2.5-flash-lite_vikhyatk+moondream2_car_square_bboxes.json'
    'union_gemini-2.5-flash-lite_vikhyatk+moondream2_cars_and_pickup_trucks_original_bboxes.json'
)
with open(annotations_file_path, 'w', encoding='utf-8') as f:
    json.dump(coco_annotations, f, ensure_ascii=False, indent=4)
    
print(f"Saved COCO annotations to: {annotations_file_path}")

### Square Bounding Boxes

In [ ]:
coco_object_categories = pd.DataFrame(
    [
        {'id': 1, 'name': 'small'}
    ]
).set_index('id').sort_index()

coco_annotations = {
    'categories': coco_object_categories.reset_index().to_dict(orient='records'),
    'images': [],
    'annotations': []
}

for i_row, row in tqdm(images_metadata.iterrows(), total=len(images_metadata)):
    image_id = len(coco_annotations['images'])

    image_data = image_dataset[row['image_file_name']]
    image = image_data['image']

    coco_image = {
        'id': image_id,
        'file_name': row['image_file_name'],
        'width': image.width,
        'height': image.height
    }
    coco_annotations['images'].append(coco_image)

    for xmin_src, ymin_src, width_src, height_src in row['annotations:bbox2d_xywh_px_reduced_union']:
        bbox_size = 1.1*max([width_src, height_src])
        x_c = xmin_src + 0.5*width_src
        y_c = ymin_src + 0.5*height_src

        xmin = x_c - 0.5*bbox_size
        ymin = y_c - 0.5*bbox_size
        width = bbox_size
        height = bbox_size

        annotation_id = len(coco_annotations['annotations'])
        coco_annotation = {
            'id': annotation_id,
            'image_id': image_id,
            'category_id': 1,
            'bbox': [round(xmin, 2), round(ymin, 2), round(width, 2), round(height, 2)],
            'area': round(width * height, 2),
            'iscrowd': 0
        }
        coco_annotations['annotations'].append(coco_annotation)


# Save Annotations
annotations_folder_path = os.path.join(image_dataset.root_folder_path, 'annotations_coco')
os.makedirs(annotations_folder_path, exist_ok=True)

annotations_file_path = os.path.join(
    annotations_folder_path,
    'union_gemini-2.5-flash-lite_vikhyatk+moondream2_cars_and_pickup_trucks_square_bboxes.json'
)
with open(annotations_file_path, 'w', encoding='utf-8') as f:
    json.dump(coco_annotations, f, ensure_ascii=False, indent=4)
    
print(f"Saved COCO annotations to: {annotations_file_path}")